In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, confusion_matrix, 
                             roc_auc_score, roc_curve, f1_score, 
                             accuracy_score, precision_score, recall_score)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from scipy.stats import uniform, randint
import time
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("XGBOOST MODEL - STANDALONE")
print("="*80)

# ============================================================================
# STEP 1: LOAD DATA
# ============================================================================

print("\nSTEP 1: Load Data & Create Target")
df = pd.read_csv('/Users/deirdreoconnor/Desktop/Ai Start Course 2/Course Materials/Week 5/Week 5 - Additional Files/Demos/Code/abt_fully_cleaned_no_nulls.csv')
print(f"Dataset: {df.shape}")

df['HasAnyCVD'] = (
    (df['HadHeartAttack'] == 'Yes') | 
    (df['HadAngina'] == 'Yes') | 
    (df['HadStroke'] == 'Yes')
).astype(int)

df = df.drop(['HadHeartAttack', 'HadAngina', 'HadStroke'], axis=1, errors='ignore')

# ============================================================================
# STEP 2: SPLIT & ENCODE
# ============================================================================

print("\nSTEP 2: Split & Encode")
X = df.drop(['HasAnyCVD', 'HasAnyCVD_cat'], axis=1)
y = df['HasAnyCVD']



X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

print(f"Features: {X_train_encoded.shape[1]}")

# Create subset
X_train_subset, _, y_train_subset, _ = train_test_split(
    X_train_encoded, y_train, train_size=0.30, random_state=42, stratify=y_train
)

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

# ============================================================================
# PHASE 1: COARSE SEARCH
# ============================================================================

print("\n" + "="*80)
print("PHASE 1: COARSE SEARCH")
print("="*80)

pipeline_xgb = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('scaler', StandardScaler()),
    ('classifier', XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False,
        n_jobs=1
    ))
])

param_dist_xgb = {
    'classifier__n_estimators': randint(50, 200),
    'classifier__max_depth': randint(3, 10),
    'classifier__learning_rate': uniform(0.01, 0.3),
    'classifier__subsample': uniform(0.6, 0.4),
    'classifier__colsample_bytree': uniform(0.6, 0.4),
    'classifier__gamma': uniform(0, 5),
    'classifier__min_child_weight': randint(1, 10)
}

random_search_xgb = RandomizedSearchCV(
    pipeline_xgb,
    param_distributions=param_dist_xgb,
    n_iter=30,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    scoring=scoring,
    refit='roc_auc',
    n_jobs=1,
    verbose=2,
    random_state=42,
    return_train_score=True
)

print("\nSearching... (20-30 minutes)")
start_time = time.time()
random_search_xgb.fit(X_train_subset, y_train_subset)
phase1_time = time.time() - start_time

print(f"\n✓ Phase 1 done! Time: {phase1_time/60:.1f} min")
print("\nBest params:")
for k, v in random_search_xgb.best_params_.items():
    print(f"  {k}: {v}")

best_idx = random_search_xgb.best_index_
print(f"\nBest CV scores:")
print(f"  Accuracy:  {random_search_xgb.cv_results_['mean_test_accuracy'][best_idx]:.4f}")
print(f"  Precision: {random_search_xgb.cv_results_['mean_test_precision'][best_idx]:.4f}")
print(f"  Recall:    {random_search_xgb.cv_results_['mean_test_recall'][best_idx]:.4f}")
print(f"  F1:        {random_search_xgb.cv_results_['mean_test_f1'][best_idx]:.4f}")
print(f"  ROC-AUC:   {random_search_xgb.cv_results_['mean_test_roc_auc'][best_idx]:.4f}")

# ============================================================================
# PHASE 2: FINE-TUNING
# ============================================================================

print("\n" + "="*80)
print("PHASE 2: FINE-TUNING")
print("="*80)

best_n_est = random_search_xgb.best_params_['classifier__n_estimators']
best_depth = random_search_xgb.best_params_['classifier__max_depth']
best_lr = random_search_xgb.best_params_['classifier__learning_rate']

param_grid_xgb = {
    'classifier__n_estimators': [best_n_est, best_n_est+20],
    'classifier__max_depth': [best_depth, best_depth+1],
    'classifier__learning_rate': [best_lr],
    'classifier__subsample': [random_search_xgb.best_params_['classifier__subsample']],
    'classifier__colsample_bytree': [random_search_xgb.best_params_['classifier__colsample_bytree']],
    'classifier__gamma': [random_search_xgb.best_params_['classifier__gamma']],
    'classifier__min_child_weight': [random_search_xgb.best_params_['classifier__min_child_weight']]
}

print(f"\nGrid: 4 combinations (20 fits)")

grid_search_xgb = GridSearchCV(
    pipeline_xgb,
    param_grid=param_grid_xgb,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring=scoring,
    refit='roc_auc',
    n_jobs=1,
    verbose=2,
    return_train_score=True
)

print("\nFine-tuning on full data... (20-30 minutes)")
start_time = time.time()
grid_search_xgb.fit(X_train_encoded, y_train)
phase2_time = time.time() - start_time

print(f"\n✓ Phase 2 done! Time: {phase2_time/60:.1f} min")
print("\nFinal best params:")
for k, v in grid_search_xgb.best_params_.items():
    print(f"  {k}: {v}")

best_idx = grid_search_xgb.best_index_
print(f"\nFinal CV scores:")
print(f"  Accuracy:  {grid_search_xgb.cv_results_['mean_test_accuracy'][best_idx]:.4f} ± {grid_search_xgb.cv_results_['std_test_accuracy'][best_idx]:.4f}")
print(f"  Precision: {grid_search_xgb.cv_results_['mean_test_precision'][best_idx]:.4f} ± {grid_search_xgb.cv_results_['std_test_precision'][best_idx]:.4f}")
print(f"  Recall:    {grid_search_xgb.cv_results_['mean_test_recall'][best_idx]:.4f} ± {grid_search_xgb.cv_results_['std_test_recall'][best_idx]:.4f}")
print(f"  F1:        {grid_search_xgb.cv_results_['mean_test_f1'][best_idx]:.4f} ± {grid_search_xgb.cv_results_['std_test_f1'][best_idx]:.4f}")
print(f"  ROC-AUC:   {grid_search_xgb.cv_results_['mean_test_roc_auc'][best_idx]:.4f} ± {grid_search_xgb.cv_results_['std_test_roc_auc'][best_idx]:.4f}")

best_xgb_model = grid_search_xgb.best_estimator_

# ============================================================================
# TEST SET EVALUATION
# ============================================================================

print("\n" + "="*80)
print("TEST SET EVALUATION")
print("="*80)

y_pred_xgb = best_xgb_model.predict(X_test_encoded)
y_pred_proba_xgb = best_xgb_model.predict_proba(X_test_encoded)[:, 1]

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=['No CVD', 'Has CVD']))

acc_xgb = accuracy_score(y_test, y_pred_xgb)
prec_xgb = precision_score(y_test, y_pred_xgb)
rec_xgb = recall_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)
auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)

print(f"\nTest Performance:")
print(f"  Accuracy:  {acc_xgb:.4f}")
print(f"  Precision: {prec_xgb:.4f}")
print(f"  Recall:    {rec_xgb:.4f}")
print(f"  F1-Score:  {f1_xgb:.4f}")
print(f"  ROC-AUC:   {auc_xgb:.4f}")

cm_xgb = confusion_matrix(y_test, y_pred_xgb)
print("\nConfusion Matrix:")
print(cm_xgb)

# ============================================================================
# SAVE RESULTS
# ============================================================================

xgb_results = pd.DataFrame({
    'Model': ['XGBoost'],
    'Accuracy': [acc_xgb],
    'Precision': [prec_xgb],
    'Recall': [rec_xgb],
    'F1-Score': [f1_xgb],
    'AUC-ROC': [auc_xgb]
})

xgb_results.to_csv('xgboost_results.csv', index=False)
print("\n✓ Saved: xgboost_results.csv")

# ============================================================================
# COMPARE WITH EXISTING MODELS
# ============================================================================

print("\n" + "="*80)
print("COMPARISON WITH EXISTING MODELS")
print("="*80)

# Your existing results
existing_results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [0.853250, 0.823321],
    'Precision': [0.372126, 0.336754],
    'Recall': [0.379324, 0.534084],
    'F1-Score': [0.375691, 0.413062],
    'AUC-ROC': [0.788475, 0.814153]
})

# Combine
all_results = pd.concat([existing_results, xgb_results], ignore_index=True)
all_results_sorted = all_results.sort_values('AUC-ROC', ascending=False)

print("\nAll Models (Ranked by AUC-ROC):")
print(all_results_sorted.to_string(index=False))

all_results_sorted.to_csv('all_models_comparison.csv', index=False)
print("\n✓ Saved: all_models_comparison.csv")

best_model = all_results_sorted.iloc[0]
print(f"\n🏆 BEST MODEL: {best_model['Model']}")
print(f"   AUC-ROC: {best_model['AUC-ROC']:.4f}")
print(f"   Recall:  {best_model['Recall']:.4f}")

print("\n" + "="*80)
print("✅ XGBoost Training Complete!")
print("="*80)






XGBOOST MODEL - STANDALONE

STEP 1: Load Data & Create Target
Dataset: (419656, 29)

STEP 2: Split & Encode
Features: 49

PHASE 1: COARSE SEARCH

Searching... (20-30 minutes)
Fitting 3 folds for each of 30 candidates, totalling 90 fits
[CV] END classifier__colsample_bytree=0.749816047538945, classifier__gamma=4.75357153204958, classifier__learning_rate=0.22959818254342154, classifier__max_depth=7, classifier__min_child_weight=5, classifier__n_estimators=152, classifier__subsample=0.7783331011414365; total time=   1.9s
[CV] END classifier__colsample_bytree=0.749816047538945, classifier__gamma=4.75357153204958, classifier__learning_rate=0.22959818254342154, classifier__max_depth=7, classifier__min_child_weight=5, classifier__n_estimators=152, classifier__subsample=0.7783331011414365; total time=   1.6s
[CV] END classifier__colsample_bytree=0.749816047538945, classifier__gamma=4.75357153204958, classifier__learning_rate=0.22959818254342154, classifier__max_depth=7, classifier__min_child_w